In [1]:
"""
Neural Atom Prevalence (NAP) — MNIS
=====================================================
Full pipeline following thesis Section 4.1.3:
  Phase 1 : Bayesian Lottery Ticket (IMP, 8 rounds, 20% prune/round)
  Phase 2 : SS-IG on winning ticket with soft Gumbel-Softmax
  Phase 3 : Poisson-Binomial optimal layer-size selection
  Phase 4 : Neural atom selection by ranked posterior inclusion prob.
  Phase 5 : Fine-tuning the sparse BNN from epoch-0 weights

Hardware  : CUDA GPU (falls back to CPU if unavailable)
Workers   : 8 DataLoader workers with pinned memory
Seeds     : fixed at 42 across PyTorch, NumPy, random
"""

import os
import math
import copy
import random
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib
matplotlib.use("Agg")          # non-interactive backend — works without a display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_WORKERS = 8


# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────

def get_loaders(dataset_name: str = "mnist",
                data_root: str = "./data",
                batch_size: int = 1024) -> tuple[DataLoader, DataLoader]:
    """
    Return (train_loader, test_loader) for MNIST or Fashion-MNIST.

    Preprocessing follows thesis Section 4.1.3 exactly:
      MNIST        : divide pixel values by 126
      Fashion-MNIST: random horizontal flip (p=0.5) + normalize
    Images are flattened to dim-784 vectors.
    """
    name = dataset_name.lower().strip()

    if name == "mnist":
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Lambda(lambda x: x / 126.0),   # thesis: divide by 126
            transforms.Lambda(lambda x: x.view(-1)),  # flatten → 784
        ])
        train_ds = datasets.MNIST(data_root, train=True,  download=True, transform=transform)
        test_ds  = datasets.MNIST(data_root, train=False, download=True, transform=transform)

    elif name in ("fashion", "fashion-mnist", "fashionmnist"):
        transform_train = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize((0.2860,), (0.3530,)),
            transforms.Lambda(lambda x: x.view(-1)),
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.2860,), (0.3530,)),
            transforms.Lambda(lambda x: x.view(-1)),
        ])
        train_ds = datasets.FashionMNIST(data_root, train=True,  download=True, transform=transform_train)
        test_ds  = datasets.FashionMNIST(data_root, train=False, download=True, transform=transform_test)
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}. Choose 'mnist' or 'fashion-mnist'.")

    loader_kwargs = dict(
        batch_size  = batch_size,
        num_workers = N_WORKERS,
        pin_memory  = True,
        persistent_workers = True,   # keep workers alive across epochs
    )
    train_loader = DataLoader(train_ds, shuffle=True,  **loader_kwargs)
    test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kwargs)

    print(f"[data] {dataset_name} | train={len(train_ds):,}  test={len(test_ds):,} "
          f"| batch={batch_size} | workers={N_WORKERS} | device={DEVICE}")
    return train_loader, test_loader


# ─────────────────────────────────────────────────────────────────────────────
# 2.  NETWORK ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────────

class SwishActivation(nn.Module):
    """f(x) = x · σ(x)  (also called SiLU — avoids dying-neuron problem)."""
    def forward(self, x):
        return x * torch.sigmoid(x)


class BayesianLinear(nn.Module):
    """
    Variational Bayesian linear layer with spike-and-slab prior.

    Each weight vector W_j | v_j ~ (1-v_j)*delta_0 + v_j * N(0, sigma0^2 I).
    Parameters learned:
      mu    : variational mean  (shape: out_features x in_features+1  for bias)
      rho   : log-sigma param   (sigma = softplus(rho))
      logit_pi : logit of inclusion probability pi_tilde_j  (shape: out_features)
    """
    def __init__(self, in_features: int, out_features: int, sigma0: float = 1.0):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.sigma0       = sigma0

        # weight + bias concatenated: shape (out, in+1)
        self.mu      = nn.Parameter(torch.empty(out_features, in_features + 1))
        self.rho     = nn.Parameter(torch.empty(out_features, in_features + 1))
        # per-neuron inclusion logit  π̃_j = sigmoid(logit_pi_j)
        self.logit_pi = nn.Parameter(torch.empty(out_features))

        self._init_params()

    def _init_params(self):
        """Kaiming uniform μ, ρ = −6 → σ ≈ 0.05, γ = 0.99 → logit ≈ 4.6."""
        nn.init.kaiming_uniform_(self.mu, a=math.sqrt(5))
        nn.init.constant_(self.rho, -6.0)
        nn.init.constant_(self.logit_pi, math.log(0.99 / 0.01))  # logit(0.99)

    # ── store epoch-0 snapshot for LTH rewinding ────────────────────────────
    def snapshot_epoch0(self):
        self._mu0       = self.mu.data.clone()
        self._rho0      = self.rho.data.clone()
        self._logit_pi0 = self.logit_pi.data.clone()

    def rewind_to_epoch0(self):
        self.mu.data.copy_(self._mu0)
        self.rho.data.copy_(self._rho0)
        self.logit_pi.data.copy_(self._logit_pi0)

    # ── inclusion probability and soft gate ─────────────────────────────────
    @property
    def pi_tilde(self) -> torch.Tensor:
        """π̃_j = sigmoid(logit_π_j)  shape: (out_features,)"""
        return torch.sigmoid(self.logit_pi)

    def gumbel_softmax_gate(self, tau: float = 0.5) -> torch.Tensor:
        """
        Soft Gumbel-Softmax gate  ṽ_j ∈ (0,1), used in BOTH forward and backward.
        ṽ_j = σ( (logit_π_j + log(α/(1-α))) / τ ),  α ~ U(0,1)
        """
        alpha   = torch.zeros_like(self.logit_pi).uniform_().clamp(1e-6, 1 - 1e-6)
        gumbel  = torch.log(alpha) - torch.log(1 - alpha)      # log-odds of U(0,1)
        return torch.sigmoid((self.logit_pi + gumbel) / tau)   # ṽ_j

    def forward(self, x: torch.Tensor, tau: float = 0.5,
                hard: bool = False) -> torch.Tensor:
        """
        Forward pass with reparameterized weights and soft gate.
        x : (batch, in_features)
        Returns : (batch, out_features)
        """
        sigma = F.softplus(self.rho)                            # (out, in+1)

        # reparameterize:  w̃ = μ + σ ⊙ ε,  ε ~ N(0,I)
        eps  = torch.randn_like(self.mu)
        w_sample = self.mu + sigma * eps                        # (out, in+1)

        w  = w_sample[:, :-1]   # weights  (out, in)
        b  = w_sample[:, -1]    # biases   (out,)

        if hard:
            # straight-through estimator (used in baseline SS-IG only)
            v = (self.pi_tilde > 0.5).float()
        else:
            # soft gate — used in NAP (both forward AND backward)
            v = self.gumbel_softmax_gate(tau)                   # (out,)

        # gate each output neuron: shape broadcast (batch, out)
        out = F.linear(x, w, b)                                 # (batch, out)
        out = out * v.unsqueeze(0)                              # gate
        return out

    # ── KL divergence terms ─────────────────────────────────────────────────
    def kl_v(self, prior_pi: float) -> torch.Tensor:
        """
        KL( q(v_j) || P(v_j) )  summed over j  — Component b of ELBO.
        = Σ_j [ π̃_j log(π̃_j/π) + (1-π̃_j) log((1-π̃_j)/(1-π)) ]
        """
        pi  = self.pi_tilde.clamp(1e-6, 1 - 1e-6)
        p   = torch.tensor(prior_pi, device=pi.device).clamp(1e-6, 1 - 1e-6)
        kl  = pi * (torch.log(pi) - torch.log(p)) + \
              (1 - pi) * (torch.log(1 - pi) - torch.log(1 - p))
        return kl.sum()

    def kl_weights(self) -> torch.Tensor:
        """
        Σ_j  π̃_j · KL( N(μ_j, diag(σ_j²)) || N(0, σ0² I) )  — Component c.
        """
        sigma   = F.softplus(self.rho)
        sigma0  = torch.tensor(self.sigma0, device=self.mu.device)
        kl_each = 0.5 * (
            (sigma / sigma0).pow(2)
            + (self.mu / sigma0).pow(2)
            - 1
            - 2 * torch.log(sigma / sigma0)
        ).sum(dim=1)                                            # (out,)
        return (self.pi_tilde * kl_each).sum()


class NAPNetwork(nn.Module):
    """
    Two-hidden-layer MLP: 784 → 400 → 400 → 10
    All hidden layers use BayesianLinear + Swish.
    Output layer is a plain deterministic linear (no gate on output neurons).
    """
    def __init__(self, hidden: int = 400, sigma0: float = 1.0):
        super().__init__()
        self.layer1 = BayesianLinear(784, hidden, sigma0)
        self.layer2 = BayesianLinear(hidden, hidden, sigma0)
        self.out    = nn.Linear(hidden, 10)
        self.act    = SwishActivation()

    def forward(self, x: torch.Tensor, tau: float = 0.5,
                hard: bool = False) -> torch.Tensor:
        x = self.act(self.layer1(x, tau=tau, hard=hard))
        x = self.act(self.layer2(x, tau=tau, hard=hard))
        return self.out(x)

    def snapshot_epoch0(self):
        self.layer1.snapshot_epoch0()
        self.layer2.snapshot_epoch0()

    def rewind_to_epoch0(self):
        self.layer1.rewind_to_epoch0()
        self.layer2.rewind_to_epoch0()
        # reinitialize output head
        nn.init.kaiming_uniform_(self.out.weight, a=math.sqrt(5))
        nn.init.zeros_(self.out.bias)

    def elbo(self, x: torch.Tensor, y: torch.Tensor,
             prior_pi1: float, prior_pi2: float,
             n_data: int, kl_weight: float = 1.0,
             tau: float = 0.5) -> torch.Tensor:
        """
        ELBO = E[log P(y|x,θ)] - kl_weight*(KL_v + KL_weights)
        Scaled by batch/n_data for mini-batch training.
        """
        logits   = self.forward(x, tau=tau, hard=False)
        log_like = -F.cross_entropy(logits, y, reduction="sum")

        kl  = self.layer1.kl_v(prior_pi1) + self.layer1.kl_weights()
        kl += self.layer2.kl_v(prior_pi2) + self.layer2.kl_weights()

        # scale likelihood to full dataset size
        scale = n_data / x.size(0)
        return scale * log_like - kl_weight * kl

    @torch.no_grad()
    def predict(self, x: torch.Tensor, n_samples: int = 10) -> torch.Tensor:
        """Monte-Carlo prediction: average softmax over n_samples forward passes."""
        probs = torch.zeros(x.size(0), 10, device=x.device)
        for _ in range(n_samples):
            probs += F.softmax(self.forward(x, tau=0.5, hard=False), dim=1)
        return probs / n_samples


# ─────────────────────────────────────────────────────────────────────────────
# 3.  PRIOR CALIBRATION  (Corollary 4.5, Jantre et al. 2022)
# ─────────────────────────────────────────────────────────────────────────────

def compute_prior_pi(k_in: int, k_out: int, k_next: int,
                     n: int, L: int,
                     B_l: float | None = None) -> float:
    """
    Compute layer-wise prior inclusion probability π_l via Corollary 4.5.

    Parameters
    ----------
    k_in   : number of input units to this layer
    k_out  : number of units in this layer  (= k_l)
    k_next : number of units in next layer  (= k_{l+1})
    n      : training set size
    L      : total number of hidden layers
    B_l    : max L1 weight norm; defaults to k_out + 1 if None (dense case)

    Returns
    -------
    pi_l   : float in (0, 1)
    """
    if B_l is None:
        B_l = float(k_out + 1)          # dense default (Appendix B.4 of [11])

    # U_m terms (use l's own dimensions as a conservative bound)
    def U_m(k):
        return (L + 1)**2 * (math.log(n) + math.log(L + 1)
                             + math.log(k + 1) + math.log(k + 1))

    # V_l computation
    U_sum = sum(U_m(k) for k in [k_in, k_out])      # simplified: sum over nearby layers
    V_l   = (B_l**2 / (k_out + 1) + math.log(B_l)  # dominant terms
             + L + math.log(k_out + 1) + math.log(n) + math.log(U_sum + 1e-8))

    C_l   = 1.0                                      # set to 1; tightened for ticket below
    log_pi = -(math.log(k_next) + C_l * (k_out + 1) * V_l)
    pi     = math.exp(max(log_pi, -20.0))            # clamp for numerical safety
    return float(np.clip(pi, 1e-6, 1 - 1e-6))


# ─────────────────────────────────────────────────────────────────────────────
# 4.  PHASE 1 — BAYESIAN LOTTERY TICKET  (IMP)
# ─────────────────────────────────────────────────────────────────────────────

def magnitude_prune(model: NAPNetwork, prune_fraction: float,
                    mask: dict | None = None) -> dict:
    """
    Global unstructured magnitude pruning.
    Returns updated binary mask dict {param_name: BoolTensor}.
    """
    # collect all weight magnitudes (exclude biases and output layer for IMP)
    all_weights, all_names = [], []
    for name, param in model.named_parameters():
        if "mu" in name and "out" not in name:
            w_mag = param.data.abs()
            if mask and name in mask:
                w_mag = w_mag * mask[name].float()   # only consider surviving weights
            all_weights.append(w_mag.flatten())
            all_names.append(name)

    flat_all  = torch.cat(all_weights)
    threshold = torch.quantile(flat_all[flat_all > 0], prune_fraction)

    new_mask = {}
    for name, param in model.named_parameters():
        if name in all_names:
            m = param.data.abs() > threshold
            if mask and name in mask:
                m = m & mask[name]
            new_mask[name] = m
        elif mask and name in mask:
            new_mask[name] = mask[name]
    return new_mask


def apply_mask(model: NAPNetwork, mask: dict):
    """Zero out masked-out weights (called after each optimizer step)."""
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in mask:
                param.data *= mask[name].float()


def count_surviving_nodes(model: NAPNetwork, mask: dict) -> dict:
    """
    For each hidden layer, count neurons with at least one surviving incoming weight.
    Returns {'layer1': S1, 'layer2': S2}.
    """
    surviving = {}
    for layer_name in ["layer1", "layer2"]:
        mu_name = f"{layer_name}.mu"
        if mu_name in mask:
            # neuron j survives if any weight in row j of mu is unmasked
            m = mask[mu_name]              # (out, in+1)
            alive = m.any(dim=1).sum().item()
        else:
            alive = getattr(model, layer_name).out_features
        surviving[layer_name] = int(alive)
    return surviving


def max_l1_norm(model: NAPNetwork, mask: dict) -> dict:
    """Max L1 weight norm per layer over surviving neurons."""
    norms = {}
    for layer_name in ["layer1", "layer2"]:
        mu_name = f"{layer_name}.mu"
        mu = getattr(model, layer_name).mu.data
        if mu_name in mask:
            mu = mu * mask[mu_name].float()
        norms[layer_name] = float(mu.abs().sum(dim=1).max().item())
    return norms


def train_one_epoch(model: NAPNetwork, loader: DataLoader,
                    optimizer: torch.optim.Optimizer,
                    prior_pi1: float, prior_pi2: float,
                    n_data: int, kl_weight: float = 1.0,
                    tau: float = 0.5, mask: dict | None = None,
                    hard: bool = False) -> float:
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = -model.elbo(x, y, prior_pi1, prior_pi2, n_data,
                           kl_weight=kl_weight, tau=tau)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
        if mask:
            apply_mask(model, mask)
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model: NAPNetwork, loader: DataLoader,
             n_samples: int = 10) -> float:
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        probs = model.predict(x, n_samples=n_samples)
        correct += probs.argmax(dim=1).eq(y).sum().item()
        total   += y.size(0)
    return correct / total


@torch.no_grad()
def evaluate_with_uncertainty(model: NAPNetwork, loader: DataLoader,
                               n_samples: int = 10,
                               n_mc_runs: int = 5) -> tuple[float, float]:
    """
    Run evaluate() n_mc_runs times (each with n_samples forward passes).
    Returns (mean_accuracy, std_accuracy) — the std captures the stochastic
    variability of the Gumbel gates and weight reparameterisation.
    """
    accs = [evaluate(model, loader, n_samples=n_samples) for _ in range(n_mc_runs)]
    return float(np.mean(accs)), float(np.std(accs))


@torch.no_grad()
def compute_predictive_uncertainty(model: NAPNetwork, loader: DataLoader,
                                    n_samples: int = 20
                                    ) -> tuple[np.ndarray, np.ndarray,
                                               np.ndarray, np.ndarray]:
    """
    Decompose predictive uncertainty via mutual information (Houlsby et al. 2011):

      Total (predictive) entropy  : H[y | x]         — full model uncertainty
      Aleatoric uncertainty       : E_θ[H[y | x, θ]] — irreducible data noise
      Epistemic uncertainty       : H[y|x] − E_θ[H]  — model / parameter uncertainty

    Returns four arrays of shape (N_test,):
        pred_entropy, epistemic, aleatoric, is_correct  (bool)
    """
    model.eval()
    all_entropy, all_epistemic, all_aleatoric, all_correct = [], [], [], []

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        # --- stack n_samples stochastic forward passes ---
        # shape: (n_samples, batch, 10)
        sample_probs = torch.stack([
            F.softmax(model(x, tau=0.5, hard=False), dim=1)
            for _ in range(n_samples)
        ], dim=0)

        mean_p = sample_probs.mean(0)                          # (batch, 10)

        # predictive (total) entropy
        pred_H  = -(mean_p * torch.log(mean_p + 1e-8)).sum(1) # (batch,)

        # aleatoric: average of per-sample entropies
        per_H   = -(sample_probs * torch.log(sample_probs + 1e-8)).sum(2)  # (n_samples, batch)
        aleat   = per_H.mean(0)                                # (batch,)

        # epistemic = mutual information
        epist   = (pred_H - aleat).clamp(min=0.0)             # (batch,)

        correct = mean_p.argmax(1).eq(y)

        all_entropy.append(pred_H.cpu().numpy())
        all_epistemic.append(epist.cpu().numpy())
        all_aleatoric.append(aleat.cpu().numpy())
        all_correct.append(correct.cpu().numpy())

    return (np.concatenate(all_entropy),
            np.concatenate(all_epistemic),
            np.concatenate(all_aleatoric),
            np.concatenate(all_correct))


def _snapshot_sparsity(model: NAPNetwork) -> dict:
    """
    Capture layer-wise π̃ statistics for plotting.
    Returns dict with mean, std, and fraction of neurons with π̃ > 0.5.
    """
    snap = {}
    for lname in ["layer1", "layer2"]:
        pi = getattr(model, lname).pi_tilde.detach().cpu().numpy()
        snap[f"{lname}_mean"] = float(pi.mean())
        snap[f"{lname}_std"]  = float(pi.std())
        snap[f"{lname}_frac_active"] = float((pi > 0.5).mean())
        snap[f"{lname}_sparsity"]    = float((pi <= 0.5).mean())
    return snap


def run_imp(model: NAPNetwork,
            train_loader: DataLoader,
            test_loader:  DataLoader,
            n_data: int,
            imp_rounds: int    = 5,
            prune_fraction: float = 0.35,
            epochs_per_round: int = 500,
            lr: float = 1e-3,
            prior_pi1: float = 0.005,
            prior_pi2: float = 0.003) -> tuple[NAPNetwork, dict, dict]:
    """
    Phase 1: Iterative Magnitude Pruning.

    Returns the model with winning-ticket weights (rewound to epoch-0),
    the final binary mask, and a history dict for plotting.
    """
    print("\n" + "="*60)
    print("PHASE 1 — Bayesian Lottery Ticket (IMP)")
    print("="*60)

    # snapshot epoch-0 initialization
    model.snapshot_epoch0()

    mask = {}
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # ── history tracking ────────────────────────────────────────────────────
    imp_history: dict = defaultdict(list)

    for rnd in range(1, imp_rounds + 1):
        print(f"\n  IMP round {rnd}/{imp_rounds}")

        # train for epochs_per_round
        for epoch in range(1, epochs_per_round + 1):
            loss = train_one_epoch(model, train_loader, optimizer,
                                   prior_pi1, prior_pi2, n_data,
                                   kl_weight=1.0, tau=0.5,
                                   mask=mask if mask else None,
                                   hard=True)   # hard gate for IMP phase
            if epoch % 100 == 0:
                acc = evaluate(model, test_loader, n_samples=5)
                print(f"    epoch {epoch:4d} | loss {loss:10.2f} | test acc {acc:.4f}")

        # record end-of-round accuracy (3 MC runs for a light uncertainty estimate)
        rnd_acc, rnd_std = evaluate_with_uncertainty(model, test_loader,
                                                     n_samples=5, n_mc_runs=3)

        # prune
        mask = magnitude_prune(model, prune_fraction, mask if mask else None)
        surviving = count_surviving_nodes(model, mask)
        total_params = sum(
            (mask[n].sum().item() if n in mask else p.numel())
            for n, p in model.named_parameters()
        )
        print(f"  → surviving nodes: {surviving}")
        print(f"  → surviving params (approx): {total_params:,}")

        imp_history["round"].append(rnd)
        imp_history["acc_mean"].append(rnd_acc)
        imp_history["acc_std"].append(rnd_std)
        imp_history["surviving_l1"].append(surviving["layer1"])
        imp_history["surviving_l2"].append(surviving["layer2"])

        # rewind weights to epoch-0
        model.rewind_to_epoch0()
        apply_mask(model, mask)

        # re-create optimizer on rewound model
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    acc = evaluate(model, test_loader, n_samples=10)
    print(f"\n[IMP done] winning ticket test acc: {acc:.4f}")
    return model, mask, dict(imp_history)


# ─────────────────────────────────────────────────────────────────────────────
# 5.  PHASE 2 — SS-IG ON WINNING TICKET WITH SOFT GATE
# ─────────────────────────────────────────────────────────────────────────────

def run_ssig_soft(model: NAPNetwork,
                  train_loader: DataLoader,
                  test_loader:  DataLoader,
                  n_data: int,
                  mask: dict,
                  surviving: dict,
                  b_norms: dict,
                  epochs: int   = 1200,
                  lr: float     = 1e-3,
                  kl_anneal_ep: int = 400,
                  L: int = 2) -> tuple[NAPNetwork, dict]:
    """
    Phase 2: run SS-IG on the winning ticket with soft Gumbel-Softmax.
    No neurons are structurally removed during this phase.
    KL annealing: weight linearly 0 → 1 over first kl_anneal_ep epochs.
    Prior π recalibrated to ticket dimensions.

    Returns (model, history_dict) for plotting.
    """
    print("\n" + "="*60)
    print("PHASE 2 — SS-IG on winning ticket (soft gate, no pruning)")
    print("="*60)

    # recalibrate prior to ticket dimensions
    s1   = surviving["layer1"]
    s2   = surviving["layer2"]
    b1   = b_norms["layer1"]
    b2   = b_norms["layer2"]

    pi1 = compute_prior_pi(784, s1, s2,   n_data, L, B_l=b1)
    pi2 = compute_prior_pi(s1,  s2, 10,   n_data, L, B_l=b2)
    print(f"  Ticket-calibrated priors: π₁={pi1:.6f}  π₂={pi2:.6f}")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # ── history tracking — every single epoch ──────────────────────────────
    N_MC_RUNS   = 3     # repeated evaluations for MC uncertainty band
    history: dict = defaultdict(list)

    for epoch in range(1, epochs + 1):
        # KL annealing weight
        kl_w = min(1.0, epoch / kl_anneal_ep)

        loss = train_one_epoch(model, train_loader, optimizer,
                               pi1, pi2, n_data,
                               kl_weight=kl_w, tau=0.5,
                               mask=mask,       # keep ticket mask active
                               hard=False)      # SOFT gate — key NAP change

        # ── evaluate every epoch (no sub-sampling) ───────────────────────────
        acc_mean, acc_std = evaluate_with_uncertainty(
            model, test_loader, n_samples=10, n_mc_runs=N_MC_RUNS)
        snap = _snapshot_sparsity(model)

        history["epoch"].append(epoch)
        history["acc_mean"].append(acc_mean)
        history["acc_std"].append(acc_std)
        for k, v in snap.items():
            history[k].append(v)

        # print every 100 epochs to keep output manageable
        if epoch % 100 == 0 or epoch == epochs:
            pi_l1 = model.layer1.pi_tilde.detach().cpu()
            pi_l2 = model.layer2.pi_tilde.detach().cpu()
            print(f"  epoch {epoch:5d} | kl_w {kl_w:.3f} | loss {loss:10.2f} "
                  f"| test acc {acc_mean:.4f} ± {acc_std:.4f} "
                  f"| mean π̃₁={pi_l1.mean():.4f}  π̃₂={pi_l2.mean():.4f}")

    return model, dict(history)


# ─────────────────────────────────────────────────────────────────────────────
# 6.  PHASE 3 — POISSON-BINOMIAL LAYER SIZE SELECTION
# ─────────────────────────────────────────────────────────────────────────────

def poisson_binomial_pmf(pi_vec: np.ndarray) -> np.ndarray:
    """
    Compute P(K = k) for k = 0, ..., n via the dynamic programming recursion
    of Hong (2013).  Complexity O(n²).

    Parameters
    ----------
    pi_vec : inclusion probabilities  shape (n,)

    Returns
    -------
    pmf : shape (n+1,)  where pmf[k] = P(K = k)
    """
    n   = len(pi_vec)
    pmf = np.zeros(n + 1, dtype=np.float64)
    pmf[0] = 1.0

    for j, p in enumerate(pi_vec):
        # iterate backwards to avoid using updated values
        for k in range(j + 1, 0, -1):
            pmf[k] = pmf[k] * (1 - p) + pmf[k - 1] * p
        pmf[0] *= (1 - p)

    return pmf


def select_optimal_layer_size(model: NAPNetwork) -> dict:
    """
    Phase 3: compute K^op_l = argmax_k P(K_l = k) for each hidden layer.
    Returns {'layer1': K1, 'layer2': K2}.
    """
    print("\n" + "="*60)
    print("PHASE 3 — Poisson-Binomial optimal layer size")
    print("="*60)

    results = {}
    for layer_name in ["layer1", "layer2"]:
        layer  = getattr(model, layer_name)
        pi_vec = layer.pi_tilde.detach().cpu().numpy().astype(np.float64)
        pi_vec = np.clip(pi_vec, 1e-8, 1 - 1e-8)

        pmf    = poisson_binomial_pmf(pi_vec)
        k_opt  = int(np.argmax(pmf))

        # ensure at least 1 neuron survives (thesis constraint)
        k_opt  = max(1, k_opt)
        results[layer_name] = k_opt

        print(f"  {layer_name}: K^op = {k_opt}  "
              f"(out of {len(pi_vec)})  "
              f"sparsity = {1 - k_opt/len(pi_vec):.2%}")
        print(f"           P(K={k_opt}) = {pmf[k_opt]:.6f}  "
              f"| mean π̃ = {pi_vec.mean():.4f}")

    return results


# ─────────────────────────────────────────────────────────────────────────────
# 7.  PHASE 4 — NEURAL ATOM SELECTION
# ─────────────────────────────────────────────────────────────────────────────

def select_atoms(model: NAPNetwork,
                 k_optimal: dict) -> dict:
    """
    Phase 4: for each hidden layer, rank neurons by π̃_j and keep top K^op_l.
    Returns {'layer1': LongTensor of selected indices,
             'layer2': LongTensor of selected indices}.
    """
    print("\n" + "="*60)
    print("PHASE 4 — Neural atom selection")
    print("="*60)

    selected = {}
    for layer_name in ["layer1", "layer2"]:
        layer  = getattr(model, layer_name)
        pi_vec = layer.pi_tilde.detach().cpu()
        k_opt  = k_optimal[layer_name]

        # rank descending — select top K^op atoms
        ranked = torch.argsort(pi_vec, descending=True)
        chosen = ranked[:k_opt].sort().values   # keep in original index order

        selected[layer_name] = chosen
        min_selected = pi_vec[chosen].min().item()
        max_rejected = (f"{pi_vec[ranked[k_opt:]].max().item():.4f}"
                        if k_opt < len(pi_vec) else "n/a")
        print(f"  {layer_name}: selected {k_opt} atoms "
              f"| min π̃ selected = {min_selected:.4f} "
              f"| max π̃ rejected = {max_rejected}")

    return selected


def build_sparse_network(selected: dict,
                         sigma0: float = 1.0) -> NAPNetwork:
    """
    Construct a new NAPNetwork with hidden sizes [K1, K2] as determined
    by atom selection.  Snapshots epoch-0 immediately so Phase 5 can rewind.
    Weights start from fresh Kaiming init — rewinding in Phase 5 restores this.
    """
    k1 = len(selected["layer1"])
    k2 = len(selected["layer2"])

    # build a minimal shell, then replace layers with exact sizes
    sparse_model = NAPNetwork(hidden=k1, sigma0=sigma0).to(DEVICE)
    sparse_model.layer1 = BayesianLinear(784, k1, sigma0).to(DEVICE)
    sparse_model.layer2 = BayesianLinear(k1,  k2, sigma0).to(DEVICE)
    sparse_model.out    = nn.Linear(k2, 10).to(DEVICE)

    # snapshot the fresh Kaiming init as "epoch-0" for LTH-style rewind in Phase 5
    sparse_model.snapshot_epoch0()
    return sparse_model


# ─────────────────────────────────────────────────────────────────────────────
# 8.  PHASE 5 — FINE-TUNING THE SPARSE BNN
# ─────────────────────────────────────────────────────────────────────────────

def run_finetune(sparse_model: NAPNetwork,
                 train_loader: DataLoader,
                 test_loader:  DataLoader,
                 n_data: int,
                 k_optimal: dict,
                 b_norms: dict,
                 epochs: int = 1200,
                 lr: float   = 1e-3,
                 L: int = 2) -> tuple[NAPNetwork, dict]:
    """
    Phase 5: fine-tune the sparse BNN from epoch-0 weights (LTH-style rewind).
    Prior is recalibrated to the sparse topology K^op.

    Returns (sparse_model, history_dict) for plotting.
    """
    print("\n" + "="*60)
    print("PHASE 5 — Fine-tuning sparse BNN from epoch-0")
    print("="*60)

    # rewind to the Kaiming init captured in build_sparse_network (LTH-style)
    sparse_model.rewind_to_epoch0()

    k1 = k_optimal["layer1"]
    k2 = k_optimal["layer2"]

    pi1 = compute_prior_pi(784, k1, k2,  n_data, L, B_l=b_norms["layer1"])
    pi2 = compute_prior_pi(k1,  k2, 10,  n_data, L, B_l=b_norms["layer2"])
    print(f"  Sparse-topology priors: π₁={pi1:.6f}  π₂={pi2:.6f}")

    optimizer = torch.optim.Adam(sparse_model.parameters(), lr=lr)

    # ── history tracking — every single epoch ──────────────────────────────
    N_MC_RUNS = 3
    history: dict = defaultdict(list)

    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(sparse_model, train_loader, optimizer,
                               pi1, pi2, n_data,
                               kl_weight=1.0, tau=0.5,
                               mask=None, hard=False)

        # ── evaluate every epoch (no sub-sampling) ───────────────────────────
        acc_mean, acc_std = evaluate_with_uncertainty(
            sparse_model, test_loader, n_samples=10, n_mc_runs=N_MC_RUNS)
        snap = _snapshot_sparsity(sparse_model)

        history["epoch"].append(epoch)
        history["acc_mean"].append(acc_mean)
        history["acc_std"].append(acc_std)
        for k, v in snap.items():
            history[k].append(v)

        # print every 100 epochs to keep output manageable
        if epoch % 100 == 0 or epoch == epochs:
            print(f"  epoch {epoch:5d} | loss {loss:10.2f} "
                  f"| test acc {acc_mean:.4f} ± {acc_std:.4f}")

    return sparse_model, dict(history)


# ─────────────────────────────────────────────────────────────────────────────
# 9.  METRICS & REPORTING
# ─────────────────────────────────────────────────────────────────────────────

def compression_report(original_hidden: int,
                        k_optimal: dict,
                        surviving: dict,
                        dataset_name: str):
    """Print layer-wise sparsity and overall compression ratio."""
    print("\n" + "="*60)
    print(f"COMPRESSION REPORT — {dataset_name.upper()}")
    print("="*60)

    k1_orig = k2_orig = original_hidden

    # original dense param count
    p_dense = 784 * k1_orig + k1_orig + k1_orig * k2_orig + k2_orig + k2_orig * 10 + 10

    k1_ticket = surviving["layer1"]
    k2_ticket = surviving["layer2"]
    k1_nap    = k_optimal["layer1"]
    k2_nap    = k_optimal["layer2"]

    p_ticket = 784 * k1_ticket + k1_ticket + k1_ticket * k2_ticket + k2_ticket + k2_ticket * 10 + 10
    p_nap    = 784 * k1_nap    + k1_nap    + k1_nap    * k2_nap    + k2_nap    + k2_nap    * 10 + 10

    print(f"\n  Dense      : 784→{k1_orig}→{k2_orig}→10   params={p_dense:,}")
    print(f"  Ticket     : 784→{k1_ticket}→{k2_ticket}→10"
          f"   params={p_ticket:,}   compression={p_dense/p_ticket:.2f}×")
    print(f"  NAP (final): 784→{k1_nap}→{k2_nap}→10"
          f"   params={p_nap:,}   compression={p_dense/p_nap:.2f}×")

    for layer_idx, (k_nap, k_orig) in enumerate(
            zip([k1_nap, k2_nap], [k1_orig, k2_orig]), start=1):
        sparsity = 1 - k_nap / k_orig
        print(f"  Layer {layer_idx} node sparsity: {sparsity:.2%} "
              f"({k_orig - k_nap} of {k_orig} nodes removed)")


# ─────────────────────────────────────────────────────────────────────────────
# 10.  FULL PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────
# 10.  VISUALISATION — UNCERTAINTY-AWARE TRAINING CURVES
# ─────────────────────────────────────────────────────────────────────────────

# ── colour palette ───────────────────────────────────────────────────────────
_C = {
    "imp"   : "#E07B39",   # warm orange
    "ssig"  : "#4C72B0",   # steel blue
    "ft"    : "#27AE60",   # emerald green
    "l1"    : "#8E44AD",   # purple  (layer 1)
    "l2"    : "#16A085",   # teal    (layer 2)
    "correct"   : "#2980B9",
    "incorrect" : "#E74C3C",
    "bg"    : "#F8F9FA",
    "grid"  : "#DEE2E6",
}

_PHASE_LABEL = {"imp": "Phase 1 — IMP",
                "ssig": "Phase 2 — SS-IG",
                "ft":   "Phase 5 — Fine-tune"}


def _shade(ax, x, mean, std, color, alpha=0.18):
    """Fill ± 1 std band around a line."""
    mean, std = np.asarray(mean), np.asarray(std)
    ax.fill_between(x, mean - std, mean + std, alpha=alpha, color=color, linewidth=0)


def _style_ax(ax, xlabel="", ylabel="", title=""):
    ax.set_facecolor(_C["bg"])
    ax.grid(True, color=_C["grid"], linewidth=0.7, linestyle="--", zorder=0)
    ax.spines[["top", "right"]].set_visible(False)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=10)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=10)
    if title:
        ax.set_title(title, fontsize=11, fontweight="bold", pad=8)


def plot_all_curves(imp_history: dict,
                    ssig_history: dict,
                    ft_history: dict,
                    uq_arrays: tuple[np.ndarray, np.ndarray,
                                     np.ndarray, np.ndarray],
                    dataset_name: str,
                    save_path: str = "nap_training_curves.png"):
    """
    Generate a 3×2 figure:

      [0,0] Accuracy vs Epoch — Phase 2 SS-IG
            Every single epoch is one dot; ± 1σ MC band overlaid.
      [0,1] Accuracy vs Epoch — Phase 5 Fine-tune
            Same treatment: every epoch a dot, ± 1σ MC band.
      [1,0] IMP — surviving nodes per round (bar chart)
      [1,1] Node sparsity (π̃ active fraction) vs Epoch (both phases)
      [2,0] Predictive entropy histogram (correct vs incorrect)
      [2,1] Reliability diagram (calibration curve) + ECE
    """
    pred_H, epist, aleat, correct = uq_arrays

    fig, axes = plt.subplots(3, 2, figsize=(16, 18), facecolor="white")
    ax_ssig  = axes[0, 0]
    ax_ft    = axes[0, 1]
    ax_imp   = axes[1, 0]
    ax_spar  = axes[1, 1]
    ax_uq    = axes[2, 0]
    ax_rel   = axes[2, 1]

    # ── helpers ───────────────────────────────────────────────────────────────
    def _shade(ax, x, mean, std, color, alpha=0.18):
        mean, std = np.asarray(mean), np.asarray(std)
        ax.fill_between(x, mean - std, mean + std,
                        alpha=alpha, color=color, linewidth=0)

    def _style_ax(ax, xlabel="", ylabel="", title=""):
        ax.set_facecolor(_C["bg"])
        ax.grid(True, color=_C["grid"], linewidth=0.7, linestyle="--", zorder=0)
        ax.spines[["top", "right"]].set_visible(False)
        if xlabel: ax.set_xlabel(xlabel, fontsize=10)
        if ylabel: ax.set_ylabel(ylabel, fontsize=10)
        if title:  ax.set_title(title, fontsize=11, fontweight="bold", pad=8)

    # ── Panel [0,0] — Accuracy every epoch, Phase 2 SS-IG ────────────────────
    _style_ax(ax_ssig,
              xlabel="Epoch",
              ylabel="Test Accuracy",
              title="A — Accuracy vs Epoch · Phase 2 SS-IG  (every epoch · ±1σ MC band)")

    ep2   = np.asarray(ssig_history["epoch"])       # one entry per epoch
    mu2   = np.asarray(ssig_history["acc_mean"])
    std2  = np.asarray(ssig_history["acc_std"])

    # every epoch = one dot (small, semi-transparent for readability)
    ax_ssig.scatter(ep2, mu2, s=4, alpha=0.35, color=_C["ssig"],
                    linewidths=0, zorder=3, label="Acc per epoch")
    # smooth ± band (fill between mu ± std — the MC uncertainty)
    _shade(ax_ssig, ep2, mu2, std2, _C["ssig"], alpha=0.25)
    # thin moving-average line so the trend is visible through the scatter
    window = max(1, len(ep2) // 40)
    ma2 = np.convolve(mu2, np.ones(window)/window, mode="same")
    ax_ssig.plot(ep2, ma2, color=_C["ssig"], linewidth=1.6,
                 alpha=0.85, zorder=4, label=f"Moving avg (w={window})")

    ax_ssig.set_ylim(max(0, mu2.min() - 0.05), min(1.0, mu2.max() + 0.02))
    ax_ssig.legend(fontsize=8, loc="lower right")

    # KL annealing boundary
    kl_end = min(400, ep2.max())
    ax_ssig.axvline(kl_end, color="#AAA", linewidth=1, linestyle=":",
                    label="KL annealing end")
    ax_ssig.text(kl_end + ep2.max()*0.01, ax_ssig.get_ylim()[0] + 0.01,
                 "KL=1", fontsize=7, color="#888")

    # ── Panel [0,1] — Accuracy every epoch, Phase 5 Fine-tune ────────────────
    _style_ax(ax_ft,
              xlabel="Epoch",
              ylabel="Test Accuracy",
              title="B — Accuracy vs Epoch · Phase 5 Fine-tune  (every epoch · ±1σ MC band)")

    ep5   = np.asarray(ft_history["epoch"])
    mu5   = np.asarray(ft_history["acc_mean"])
    std5  = np.asarray(ft_history["acc_std"])

    ax_ft.scatter(ep5, mu5, s=4, alpha=0.35, color=_C["ft"],
                  linewidths=0, zorder=3, label="Acc per epoch")
    _shade(ax_ft, ep5, mu5, std5, _C["ft"], alpha=0.25)
    window5 = max(1, len(ep5) // 40)
    ma5 = np.convolve(mu5, np.ones(window5)/window5, mode="same")
    ax_ft.plot(ep5, ma5, color=_C["ft"], linewidth=1.6,
               alpha=0.85, zorder=4, label=f"Moving avg (w={window5})")
    ax_ft.set_ylim(max(0, mu5.min() - 0.05), min(1.0, mu5.max() + 0.02))
    ax_ft.legend(fontsize=8, loc="lower right")

    # ── Panel [1,0] — IMP surviving nodes ────────────────────────────────────
    _style_ax(ax_imp, xlabel="IMP Round", ylabel="Surviving Nodes",
              title="C — Surviving Nodes per IMP Round")

    width = 0.35
    r  = np.asarray(imp_history["round"])
    s1 = np.asarray(imp_history["surviving_l1"])
    s2 = np.asarray(imp_history["surviving_l2"])
    ax_imp.bar(r - width/2, s1, width, color=_C["l1"], alpha=0.85,
               label="Layer 1", zorder=3)
    ax_imp.bar(r + width/2, s2, width, color=_C["l2"], alpha=0.85,
               label="Layer 2", zorder=3)
    ax_imp.set_xticks(r)
    ax_imp.legend(fontsize=9)
    for xi, y1, y2 in zip(r, s1, s2):
        ax_imp.text(xi - width/2, y1 + 1, str(int(y1)),
                    ha="center", va="bottom", fontsize=7, color=_C["l1"])
        ax_imp.text(xi + width/2, y2 + 1, str(int(y2)),
                    ha="center", va="bottom", fontsize=7, color=_C["l2"])

    # ── Panel [1,1] — Node Sparsity vs Epoch ─────────────────────────────────
    _style_ax(ax_spar,
              xlabel="Epoch",
              ylabel="Fraction of Neurons Active  (π̃ > 0.5)",
              title="D — Node Sparsity vs Epoch (± 1σ from π̃ spread)")

    def _plot_phase_sparsity(ax, hist, color_l1, color_l2, label_prefix):
        ep  = np.asarray(hist["epoch"])
        f1  = np.asarray(hist["layer1_frac_active"])
        f2  = np.asarray(hist["layer2_frac_active"])
        s1_ = np.asarray(hist["layer1_std"])
        s2_ = np.asarray(hist["layer2_std"])
        ax.plot(ep, f1, color=color_l1, linewidth=1.8,
                label=f"{label_prefix} L1")
        _shade(ax, ep, f1, s1_, color_l1)
        ax.plot(ep, f2, color=color_l2, linewidth=1.8, linestyle="--",
                label=f"{label_prefix} L2")
        _shade(ax, ep, f2, s2_, color_l2)

    _plot_phase_sparsity(ax_spar, ssig_history, _C["ssig"], "#7EB3E0", "SS-IG")
    _plot_phase_sparsity(ax_spar, ft_history,   _C["ft"],   "#76D7B0", "FT")
    ax_spar.set_ylim(-0.05, 1.05)
    ax_spar.legend(fontsize=8, loc="upper right", ncol=2)
    ax_spar.axvline(x=ssig_history["epoch"][-1], color="#AAA",
                    linewidth=1, linestyle=":", zorder=1)
    ax_spar.text(ssig_history["epoch"][-1] * 0.98, 0.02,
                 "← SS-IG | FT →", fontsize=7, color="#777", ha="right")

    # ── Panel [2,0] — Predictive entropy histogram ────────────────────────────
    _style_ax(ax_uq,
              xlabel="Predictive Entropy  H[y | x]",
              ylabel="Density",
              title="E — Predictive Uncertainty Decomposition (Final Model)")

    bins = np.linspace(0, pred_H.max() * 1.05, 50)
    ax_uq.hist(pred_H[correct],  bins=bins, density=True, alpha=0.6,
               color=_C["correct"],   label="Correct predictions",  zorder=3)
    ax_uq.hist(pred_H[~correct], bins=bins, density=True, alpha=0.6,
               color=_C["incorrect"], label="Incorrect predictions", zorder=3)
    ax_uq.legend(fontsize=8)

    # inset: epistemic vs aleatoric scatter
    ax_in = ax_uq.inset_axes([0.57, 0.42, 0.40, 0.52])
    ax_in.set_facecolor(_C["bg"])
    rng = np.random.default_rng(42)
    idx = rng.choice(len(pred_H), size=min(2000, len(pred_H)), replace=False)
    ax_in.scatter(aleat[idx], epist[idx],
                  c=np.where(correct[idx], _C["correct"], _C["incorrect"]),
                  alpha=0.3, s=4, linewidths=0)
    ax_in.set_xlabel("Aleatoric", fontsize=7)
    ax_in.set_ylabel("Epistemic", fontsize=7)
    ax_in.set_title("Epist. vs Aleat.", fontsize=7)
    ax_in.tick_params(labelsize=6)
    ax_in.spines[["top", "right"]].set_visible(False)

    mean_H_c   = pred_H[correct].mean()
    mean_H_w   = pred_H[~correct].mean() if (~correct).sum() > 0 else 0.0
    mean_epist = epist.mean()
    mean_aleat = aleat.mean()
    ax_uq.text(0.02, 0.97,
               (f"Correct  mean H = {mean_H_c:.3f}\n"
                f"Wrong    mean H = {mean_H_w:.3f}\n"
                f"Epistemic mean  = {mean_epist:.3f}\n"
                f"Aleatoric mean  = {mean_aleat:.3f}"),
               transform=ax_uq.transAxes, fontsize=8,
               verticalalignment="top",
               bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                         edgecolor="#CCC", alpha=0.9))

    # ── Panel [2,1] — Reliability diagram (calibration) ──────────────────────
    _style_ax(ax_rel,
              xlabel="Mean Confidence (max p̄_c)",
              ylabel="Fraction Correct",
              title="F — Reliability Diagram (Calibration)")

    # confidence = max of mean predictive probability per sample
    # recompute from stored uq arrays
    # pred_H already computed; we need confidence = 1 - uncertainty proxy
    # best proxy available: use aleatoric as surrogate for low confidence
    # — but we actually need mean_p.max per sample.
    # We reuse the epistemic / aleatoric decomposition to build bins.
    # Since we don't store raw probs here, approximate confidence as
    # exp(-pred_H)  which is monotone in confidence for softmax outputs.
    confidence = np.exp(-pred_H)   # monotone proxy; exact shape preserved

    n_bins    = 15
    bin_edges = np.linspace(confidence.min(), confidence.max(), n_bins + 1)
    bin_conf, bin_acc, bin_cnt = [], [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        in_bin = (confidence >= lo) & (confidence < hi)
        if in_bin.sum() > 0:
            bin_conf.append(confidence[in_bin].mean())
            bin_acc.append(correct[in_bin].mean())
            bin_cnt.append(in_bin.sum())

    bin_conf = np.array(bin_conf)
    bin_acc  = np.array(bin_acc)
    bin_cnt  = np.array(bin_cnt)
    gap      = np.abs(bin_conf - bin_acc)

    # colour bars by calibration gap (green = well-calibrated, red = poor)
    cmap  = plt.cm.RdYlGn_r
    norm_gap = gap / (gap.max() + 1e-8)
    bar_w = (bin_edges[1] - bin_edges[0]) * 0.85
    for xc, ya, ng in zip(bin_conf, bin_acc, norm_gap):
        ax_rel.bar(xc, ya, width=bar_w, color=cmap(ng), alpha=0.85, zorder=3)

    # perfect calibration line
    lim = [bin_conf.min() - 0.02, bin_conf.max() + 0.02]
    ax_rel.plot(lim, lim, "k--", linewidth=1.3, label="Perfect calibration", zorder=4)
    ax_rel.set_xlim(lim);  ax_rel.set_ylim(0, 1.05)

    # ECE (Expected Calibration Error)
    weights = bin_cnt / bin_cnt.sum()
    ece     = float((weights * gap).sum())
    # MCE (Maximum Calibration Error)
    mce     = float(gap.max())

    ax_rel.text(0.04, 0.93,
                f"ECE = {ece:.4f}\nMCE = {mce:.4f}",
                transform=ax_rel.transAxes, fontsize=9,
                verticalalignment="top",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                          edgecolor="#CCC", alpha=0.9))

    # secondary bar showing sample counts per bin
    ax_rel2 = ax_rel.twinx()
    ax_rel2.bar(bin_conf, bin_cnt, width=bar_w, alpha=0.15,
                color="#555", zorder=1)
    ax_rel2.set_ylabel("Samples per bin", fontsize=8, color="#555")
    ax_rel2.tick_params(axis="y", colors="#555", labelsize=8)
    ax_rel2.spines[["top"]].set_visible(False)

    ax_rel.legend(fontsize=8, loc="upper left")

    # colorbar legend for calibration gap
    sm = plt.cm.ScalarMappable(cmap=cmap,
                               norm=plt.Normalize(vmin=0, vmax=gap.max()))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax_rel, fraction=0.03, pad=0.12)
    cbar.set_label("Calibration gap |conf − acc|", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    # ── suptitle & save ───────────────────────────────────────────────────────
    fig.suptitle(
        f"Neural Atom Prevalence (NAP) — {dataset_name.upper()}\n"
        f"Uncertainty-Aware Training Diagnostics",
        fontsize=14, fontweight="bold", y=0.995
    )
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\n[plot] saved → {save_path}")
    return save_path


def run_nap_pipeline(dataset_name: str = "mnist"):

    # ── hyperparameters (Section 4.1.3) ─────────────────────────────────────
    HIDDEN        = 400
    BATCH_SIZE    = 1024
    LR            = 1e-3
    SIGMA0        = 1.0
    TAU           = 0.5
    MC_SAMPLES    = 10
    KL_ANNEAL_EP  = 400     # first 1/3 of 1200 epochs
    # IMP
    IMP_ROUNDS    = 5
    PRUNE_FRAC    = 0.35
    EPOCHS_IMP    = 500
    PRIOR_PI1_IMP = 0.005   # dense-architecture priors for IMP phase
    PRIOR_PI2_IMP = 0.003
    # SS-IG + fine-tune
    EPOCHS_SSIG   = 1200
    EPOCHS_FT     = 1200
    L             = 2       # number of hidden layers

    # ── data ────────────────────────────────────────────────────────────────
    train_loader, test_loader = get_loaders(dataset_name,
                                            batch_size=BATCH_SIZE)
    n_data = len(train_loader.dataset)

    # ── build and move model to GPU ─────────────────────────────────────────
    model = NAPNetwork(hidden=HIDDEN, sigma0=SIGMA0).to(DEVICE)
    print(f"\n[model] {sum(p.numel() for p in model.parameters()):,} parameters on {DEVICE}")

    # ── Phase 1: IMP ─────────────────────────────────────────────────────────
    model, mask, imp_history = run_imp(
        model, train_loader, test_loader, n_data,
        imp_rounds    = IMP_ROUNDS,
        prune_fraction= PRUNE_FRAC,
        epochs_per_round = EPOCHS_IMP,
        lr            = LR,
        prior_pi1     = PRIOR_PI1_IMP,
        prior_pi2     = PRIOR_PI2_IMP,
    )

    surviving = count_surviving_nodes(model, mask)
    b_norms   = max_l1_norm(model, mask)
    print(f"\n[ticket] surviving nodes: {surviving}")
    print(f"[ticket] max L1 norms   : {b_norms}")

    # ── Phase 2: SS-IG soft on ticket ────────────────────────────────────────
    model, ssig_history = run_ssig_soft(
        model, train_loader, test_loader, n_data,
        mask       = mask,
        surviving  = surviving,
        b_norms    = b_norms,
        epochs     = EPOCHS_SSIG,
        lr         = LR,
        kl_anneal_ep = KL_ANNEAL_EP,
        L          = L,
    )

    # ── Phase 3: Poisson-Binomial size selection ──────────────────────────────
    k_optimal = select_optimal_layer_size(model)

    # ── Phase 4: atom selection ───────────────────────────────────────────────
    selected = select_atoms(model, k_optimal)

    # ── Phase 5: build sparse model and fine-tune ─────────────────────────────
    sparse_model = build_sparse_network(selected, sigma0=SIGMA0)
    sparse_model, ft_history = run_finetune(
        sparse_model, train_loader, test_loader, n_data,
        k_optimal = k_optimal,
        b_norms   = b_norms,
        epochs    = EPOCHS_FT,
        lr        = LR,
        L         = L,
    )

    # ── Final evaluation ──────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("FINAL EVALUATION")
    print("="*60)
    final_acc = evaluate(sparse_model, test_loader, n_samples=MC_SAMPLES)
    print(f"  NAP sparse model test accuracy: {final_acc:.4f}")

    # ── Predictive uncertainty on final model ─────────────────────────────────
    print("\n[UQ] computing predictive uncertainty on test set …")
    uq_arrays = compute_predictive_uncertainty(sparse_model, test_loader,
                                               n_samples=20)
    pred_H, epist, aleat, correct = uq_arrays
    print(f"  Mean predictive entropy : {pred_H.mean():.4f}")
    print(f"  Epistemic (mean)        : {epist.mean():.4f}")
    print(f"  Aleatoric (mean)        : {aleat.mean():.4f}")
    print(f"  Accuracy (UQ pass)      : {correct.mean():.4f}")

    compression_report(HIDDEN, k_optimal, surviving, dataset_name)

    # ── Training-curve plots ──────────────────────────────────────────────────
    plot_path = f"nap_{dataset_name}_training_curves.png"
    plot_all_curves(imp_history, ssig_history, ft_history,
                    uq_arrays, dataset_name, save_path=plot_path)

    # ── Save checkpoint ───────────────────────────────────────────────────────
    ckpt_path = f"nap_{dataset_name}_sparse_model.pt"
    torch.save({
        "state_dict"   : sparse_model.state_dict(),
        "k_optimal"    : k_optimal,
        "surviving"    : surviving,
        "b_norms"      : b_norms,
        "final_acc"    : final_acc,
        "selected"     : {k: v.tolist() for k, v in selected.items()},
        "imp_history"  : imp_history,
        "ssig_history" : ssig_history,
        "ft_history"   : ft_history,
        "uq_summary"   : {
            "mean_pred_entropy" : float(pred_H.mean()),
            "mean_epistemic"    : float(epist.mean()),
            "mean_aleatoric"    : float(aleat.mean()),
            "uq_accuracy"       : float(correct.mean()),
        },
    }, ckpt_path)
    print(f"\n[saved] checkpoint → {ckpt_path}")
    print(f"[saved] plot       → {plot_path}")

    return sparse_model, k_optimal, final_acc


# ─────────────────────────────────────────────────────────────────────────────
# 11.  ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────


DATASET = "mnist"   
# ────────────────────────────────────────────────────────────────────────────

import argparse as _argparse
import sys as _sys

def _parse_dataset() -> str:
    """
    Return the dataset name whether the script is run:
      - directly          : python nap_mnist.py --dataset fashion-mnist
      - in Jupyter        : uses the DATASET variable above
      - via ipykernel     : uses the DATASET variable above
    """
    # detect Jupyter / ipykernel environment
    in_jupyter = (
        "ipykernel" in _sys.modules
        or "IPython" in _sys.modules
        or any("jupyter" in a or "ipykernel" in a for a in _sys.argv)
    )
    if in_jupyter:
        return DATASET

    parser = _argparse.ArgumentParser(description="NAP pipeline — MNIST / Fashion-MNIST")
    parser.add_argument("--dataset", type=str, default=DATASET,
                        choices=["mnist", "fashion-mnist"],
                        help="Dataset to use (default: mnist)")
    args, _ = parser.parse_known_args()   # parse_known_args ignores unknown flags
    return args.dataset


if __name__ == "__main__":
    dataset = _parse_dataset()

    print(f"\n{'='*60}")
    print(f"  Neural Atom Prevalence — {dataset.upper()}")
    print(f"  Device : {DEVICE}")
    print(f"  Workers: {N_WORKERS}")
    print(f"{'='*60}\n")

    run_nap_pipeline(dataset_name=dataset)


  Neural Atom Prevalence — MNIST
  Device : cuda
  Workers: 8

Failed to download (trying next):
HTTP Error 404: Not Found



100%|███████████████████████████████████████| 9.91M/9.91M [00:51<00:00, 194kB/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 92.2kB/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|███████████████████████████████████████| 1.65M/1.65M [00:06<00:00, 257kB/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 3.07MB/s]

Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw

[data] mnist | train=60,000  test=10,000 | batch=1024 | workers=8 | device=cuda

[model] 953,610 parameters on cuda

PHASE 1 — Bayesian Lottery Ticket (IMP)

  IMP round 1/5


    epoch  100 | loss  229973.57 | test acc 0.9120
    epoch  200 | loss   52066.71 | test acc 0.9050
    epoch  300 | loss   41541.37 | test acc 0.9104
    epoch  400 | loss   38722.21 | test acc 0.9088
    epoch  500 | loss   36850.09 | test acc 0.9083
  → surviving nodes: {'layer1': 400, 'layer2': 400}
  → surviving params (approx): 787,570

  IMP round 2/5
    epoch  100 | loss  225336.61 | test acc 0.9158
    epoch  200 | loss   55036.48 | test acc 0.9072
    epoch  300 | loss   45262.47 | test acc 0.9075
    epoch  400 | loss   41087.31 | test acc 0.9103
    epoch  500 | loss   37389.96 | test acc 0.9127
  → surviving nodes: {'layer1': 400, 'layer2': 400}
  → surviving params (approx): 679,644

  IMP round 3/5
    epoch  100 | loss  239570.80 | test acc 0.9123
    epoch  200 | loss   57248.30 | test acc 0.9026
    epoch  300 | loss   46074.80 | test acc 0.9094
    epoch  400 | loss   41469.38 | test acc 0.9083
    epoch  500 | loss   37895.60 | test acc 0.9112
  → surviving nodes

In [3]:
import os
os.getcwd() 

'/home/student25/Desktop/Thesis code'

In [7]:
# Load from checkpoint instead of retraining
ckpt = torch.load("nap_mnist_sparse_model.pt", map_location=DEVICE)
k_optimal = ckpt["k_optimal"]

sparse_model = NAPNetwork(hidden=k_optimal["layer1"]).to(DEVICE)
sparse_model.layer1 = BayesianLinear(784, k_optimal["layer1"]).to(DEVICE)
sparse_model.layer2 = BayesianLinear(k_optimal["layer1"], k_optimal["layer2"]).to(DEVICE)
sparse_model.out    = nn.Linear(k_optimal["layer2"], 10).to(DEVICE)
sparse_model.load_state_dict(ckpt["state_dict"])
sparse_model.eval()

_, test_loader = get_loaders("mnist", batch_size=1024)

print("Model and data loaded. k_optimal:", k_optimal)

[data] mnist | train=60,000  test=10,000 | batch=1024 | workers=8 | device=cuda
Model and data loaded. k_optimal: {'layer1': 21, 'layer2': 38}


/tmp/ipykernel_36652/3005833916.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load("nap_mnist_sparse_model.pt", map_location=DEVICE)


In [8]:
@torch.no_grad()
def zero_input_accuracy(model, loader, n_samples=10):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x_zeros = torch.zeros_like(x).to(DEVICE)
        y = y.to(DEVICE)
        probs = model.predict(x_zeros, n_samples=n_samples)
        correct += probs.argmax(1).eq(y).sum().item()
        total   += y.size(0)
    return correct / total

@torch.no_grad()
def shuffled_label_sanity(model, loader, n_samples=10):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x = x.to(DEVICE)
        y_shuffled = y[torch.randperm(y.size(0))].to(DEVICE)
        probs = model.predict(x, n_samples=n_samples)
        correct += probs.argmax(1).eq(y_shuffled).sum().item()
        total   += y.size(0)
    return correct / total

In [9]:
normal_acc   = evaluate(sparse_model, test_loader, n_samples=10)
zero_acc     = zero_input_accuracy(sparse_model, test_loader)
shuffled_acc = shuffled_label_sanity(sparse_model, test_loader)

print(f"  Normal accuracy        : {normal_acc:.4f}")
print(f"  Zero-input accuracy    : {zero_acc:.4f}  (expect ~0.10 if data is used)")
print(f"  Shuffled-label accuracy: {shuffled_acc:.4f}  (expect ~0.10 if data is used)")
print(f"  Data-use gain          : {normal_acc - zero_acc:+.4f}  above zero-input baseline")

  Normal accuracy        : 0.9643
  Zero-input accuracy    : 0.0892  (expect ~0.10 if data is used)
  Shuffled-label accuracy: 0.0985  (expect ~0.10 if data is used)
  Data-use gain          : +0.8751  above zero-input baseline
